# 07.4 — Loss weights curriculum

The second curriculum lever schedules the **loss weights** — the per-component multipliers that feed the multi-objective sum ([06.1](../06_loss_orchestration/06.1_multi_task_losses_overview.ipynb), [06.4](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)). Instead of fixed weights, the Optimal regime *ramps them over training* to enact "reconstruct first, classify later": early epochs weight reconstruction heavily and classification near-zero; later epochs ramp classification up and let reconstruction decay. And the KL weight gets a *second* layer of scheduling — an annealing ramp — to prevent posterior collapse. This notebook is that lever.

This notebook covers:

* How scheduled weights feed the EMA-normalized aggregator.
* The reconstruct-first → classify-later weight shape, drawn.
* KL annealing's *two* mechanisms: the base-anneal and the dynamic multiply.
* Per-epoch snapshotting (vs the load lever's live-read).

**Prerequisites:** [07.2 piecewise schedules](07.2_piecewise_linear_schedules.ipynb), [06.4 EMA prior normalization](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb).

## Section 1 — What MATLAB does

`cgg_generateLossWeights_v2` builds per-epoch `WeightKL`, `WeightClassification`, etc., and `cgg_annealWeight` adds a ramp on the KL base:

```matlab
LossWeights = cgg_generateLossWeights_v2(...);
W_class = cgg_calculateDynamicValue(LossWeights.Classification, epoch);  % dynamic multiply
W_recon = cgg_calculateDynamicValue(LossWeights.Reconstruction, epoch);

% KL gets an EXTRA base-anneal on top of its dynamic weight:
WeightKL_base = cgg_annealWeight(epoch, WeightKL, WeightDelayEpoch, WeightEpochRamp);
W_kl = cgg_calculateDynamicValue(LossWeights.KL_with_base(WeightKL_base), epoch);
```

The port maps these to `make_weight_schedule(reconstruction=, kl=, classification=, confidence=, offset_and_scale=)` for the dynamic multipliers, and `KLBaseAnneal` for the legacy KL base ramp. `NaN` disables a component (it's skipped from the gradient root-sum, [06.11](../06_loss_orchestration/06.11_single_total_loss_three_subnetworks.ipynb)).

## Section 2 — The Python concepts you need

### 2.1 — Scheduled weights feed the normalized aggregator

Recall the loss pipeline ([06.4](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)): each component is EMA-normalized to a common scale, then multiplied by its **weight**, then summed. The weights are what set the *balance* among reconstruction, KL, classification, confidence, and offset/scale. Making those weights *time-varying* means the balance itself shifts over training — that's the whole curriculum idea, applied to the objective. The scheduled weight for epoch N is exactly the `weights` dict passed into `aggregate_normalized_losses`.

### 2.2 — Reconstruct first, classify later

In [ ]:
from neural_data_decoding.training.schedules.factory import make_weight_schedule, ScheduleWaypoints

# The Soft Three-Stage weight curriculum:
#   classification: near-zero until epoch 10, ramp to full by 15   [0,10,15] → [1e-2, 1e-2, 1.0]
#   reconstruction & kl: full until epoch 20, decay by 30          [20,30]   → [1.0, 1e-2]
waypoints = {
    "classification": ScheduleWaypoints.of((0, 10, 15), (1e-2, 1e-2, 1.0)),
    "reconstruction": ScheduleWaypoints.of((20, 30), (1.0, 1e-2)),
    "kl":             ScheduleWaypoints.of((20, 30), (1.0, 1e-2)),
}
sched = make_weight_schedule(reconstruction=1.0, kl=1.0, classification=1.0, waypoints=waypoints)

print("epoch | recon |  kl   | class  |  phase")
for e in (1, 10, 15, 20, 25, 30, 35):
    sched.update(e)
    r, k, c = sched.current("reconstruction"), sched.current("kl"), sched.current("classification")
    phase = "reconstruct" if c < 0.5 else ("handoff" if r > 0.5 else "classify")
    print(f"  {e:2}  | {r:.3f} | {k:.3f} | {c:.3f}  |  {phase}")

Read the phases down the table:

* **Epochs 1–10 (reconstruct):** classification weight is `0.01` — essentially off. The encoder/decoder learn to reconstruct the neural signal with almost no pressure to classify. This is the loss-weight version of Stage 1 ([05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb)).
* **Epochs 10–20 (handoff):** classification ramps up to full while reconstruction/KL are still full. The classifier starts learning on an *already-useful* latent.
* **Epochs 20–30+ (classify):** reconstruction and KL *decay* toward `0.01`, so the objective shifts to prioritize the decoding task now that the representation is established.

The weights choreograph exactly the pedagogy [07.1](07.1_curriculum_learning_intuition.ipynb) argued for — and they do it *smoothly*, via ramps, rather than a hard stage switch.

### 2.3 — Draw the handoff

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, 36))
curves = {"reconstruction": [], "kl": [], "classification": []}
for e in epochs:
    sched.update(e)
    for name in curves:
        curves[name].append(sched.current(name))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, curves["reconstruction"], "o-", label="reconstruction", color="#4a7a1a", markersize=4)
ax.plot(epochs, curves["kl"], "s--", label="KL", color="#1a4a7a", markersize=4)
ax.plot(epochs, curves["classification"], "d-", label="classification", color="crimson", markersize=4)
ax.axvspan(1, 10, alpha=0.08, color="green"); ax.text(5, 1.05, "reconstruct", ha="center", fontsize=9)
ax.axvspan(10, 20, alpha=0.08, color="orange"); ax.text(15, 1.05, "handoff", ha="center", fontsize=9)
ax.axvspan(20, 35, alpha=0.08, color="red"); ax.text(27, 1.05, "classify", ha="center", fontsize=9)
ax.set_xlabel("epoch"); ax.set_ylabel("loss weight"); ax.set_title("Reconstruct-first → classify-later: the weight handoff")
ax.legend(loc="center right")
plt.tight_layout(); plt.show()
print("Classification (red) rises as reconstruction/KL (green/blue) fall — a smooth objective handoff.")

### 2.4 — KL annealing: two mechanisms stacked

The KL weight is special — it gets scheduled *twice*, and both layers matter:

1. **The dynamic multiply** (§2.2): the `kl` entry's waypoints, interpolated per epoch like any other weight.
2. **The base-anneal** (`KLBaseAnneal`): a *separate* ramp on the KL weight's *base value*, ramping from 0 up to the configured `WeightKL` over `[delay_epoch, delay_epoch + epoch_ramp]`. The bundle rewrites `weight["kl"].base` from this ramp *before* applying the dynamic multiply.

So the effective KL weight is `base_anneal(epoch) × dynamic_magnitude(epoch)`. Why two? The base-anneal is the classic beta-VAE warmup that fights posterior collapse ([07.1 §2.2](07.1_curriculum_learning_intuition.ipynb)) — hold KL near zero early, ramp it in. The dynamic multiply is the *curriculum's* separate say over the KL weight (e.g. decaying it late). They compose.

In [ ]:
from neural_data_decoding.training.schedules.bundle import KLBaseAnneal

# Base-anneal: ramp the KL base from 0 to 1.0 over epochs [5, 10].
kl_anneal = KLBaseAnneal(initial_weight=1.0, delay_epoch=5, epoch_ramp=5)
print("epoch | KL base-anneal | (the beta-VAE warmup ramp on the base)")
for e in range(1, 13):
    print(f"  {e:2}  |     {kl_anneal.value_at(e):.2f}       |  {'·' * int(kl_anneal.value_at(e) * 20)}")
print("→ KL base held at 0 through the warmup, then ramped in — then the dynamic multiply scales it further.")

### 2.5 — Per-epoch snapshot, not live-read

Unlike the load lever ([07.3 §2.4](07.3_load_parameters.ipynb)), which is read *live* inside `__getitem__`, the loss weights are **snapshotted once per epoch** into a plain dict (`_resolve_epoch_loss_weights`) that the whole epoch's batches share. Why the difference? Augmentation is per-*sample* (each trial gets its own noise draw), so it must be read per-batch; loss weights are per-*epoch* (the balance shouldn't change mid-epoch), so they're resolved once. This matches MATLAB, where the weights are computed at the top of each epoch's loop. Both levers ride the same interpolator; they differ only in *when* the value is read.

## Section 3 — The `neural_data_decoding` implementation

`make_weight_schedule` — the five component weights, NaN/zero defaults:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/schedules/factory.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "def make_weight_schedule" in line)
for k in range(i, i + 9):
    print(f"{k + 1:4} | {src[k]}")
print("...")
# the per-epoch snapshot resolver:
lc = Path("../../src/neural_data_decoding/training/lifecycle.py").read_text().splitlines()
j0 = next(j for j, line in enumerate(lc) if "def _resolve_epoch_loss_weights" in line)
for k in range(j0, j0 + 3):
    print(f"{k + 1:4} | {lc[k]}")

Things to spot:

* `make_weight_schedule` defaults: `reconstruction`, `kl`, `classification` are `NaN` (disabled unless configured); `confidence` and `offset_and_scale` default to `0.0` (present but zero-weighted). `NaN` means "skip this component from the gradient sum" ([06.11](../06_loss_orchestration/06.11_single_total_loss_three_subnetworks.ipynb)).
* `_resolve_epoch_loss_weights(static_weights, curriculum)` starts from the static weights, then overrides each name with `curriculum.weight.current(name)` — the per-epoch snapshot (§2.5).
* `KLBaseAnneal.value_at` (in `bundle.py`) is `piecewise_anneal_value(initial_weight, (delay, delay+ramp), (0, 1), epoch)` — the same interpolator ([07.2](07.2_piecewise_linear_schedules.ipynb)), with the same off-by-one; `CurriculumBundle.update` applies it to `weight["kl"].base` before the dynamic multiply (§2.4).
* The resolved weight dict flows into `aggregate_normalized_losses(..., weights=epoch_loss_weights)` ([06.4 §3](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)) — the curriculum's output *is* that function's `weights` argument.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the classification-off phase

Confirm that during the reconstruct phase (epochs 1–10) the classification weight stays at its floor while reconstruction is at full.

In [ ]:
# Reveal:
for e in (1, 5, 10):
    sched.update(e)
    print(f"epoch {e:2}: classification={sched.current('classification'):.3f} (floored), "
          f"reconstruction={sched.current('reconstruction'):.3f} (full)")
print("→ near-zero classification weight = 'don't classify yet'; the encoder learns to reconstruct first.")

### Exercise 4.2 — the stacked KL weight

Compute the *effective* KL weight (base-anneal × dynamic multiply) at a few epochs, using the base-anneal from §2.4 and a dynamic `kl` schedule that holds at 1.0 then decays.

In [ ]:
# Reveal:
kl_dynamic = make_weight_schedule(kl=1.0, waypoints={"kl": ScheduleWaypoints.of((20, 30), (1.0, 0.1))})
for e in (3, 10, 25):
    kl_dynamic.update(e)
    base = kl_anneal.value_at(e)         # the warmup ramp on the base
    dyn = kl_dynamic.current("kl")       # the dynamic multiply
    print(f"epoch {e:2}: base-anneal={base:.2f} × dynamic={dyn:.2f} = effective KL weight {base * dyn:.3f}")
print("→ early: base-anneal keeps it ~0 (anti-collapse); late: dynamic decays it. Two knobs, one weight.")

## Section 5 — Common errors

### Classification never trains (weight stuck at floor)

The curriculum may still be in the reconstruct phase, or the classification waypoints never ramp up. Check the schedule reaches `1.0` (§2.2) — a floor of `1e-2` is *near*-off but the ramp must fire.

### Posterior collapse despite a KL schedule

The dynamic multiply alone may ramp KL in too fast. The *base-anneal* (§2.4) is the dedicated anti-collapse warmup — ensure `weight_epoch_ramp > 0` so `KLBaseAnneal` is active, not just the dynamic weight.

### Weights change mid-epoch

They shouldn't — they're snapshotted once per epoch (§2.5). If you see intra-epoch weight changes, something is calling `schedule.update` or re-resolving mid-epoch. Resolve once at the top.

### A NaN weight silently drops a component

`NaN` means "skip from the gradient sum" ([06.11](../06_loss_orchestration/06.11_single_total_loss_three_subnetworks.ipynb)). If a loss component seems to have no effect, its weight may be NaN rather than a small number. `0.0` keeps it present-but-zero; `NaN` removes it entirely.

### Curriculum weights fight the EMA normalization

The weights act on the *post-normalization* scale ([06.4 §5](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)). A weight ramp interacts with the normalization's own rebalancing ([06.12 §2.4](../06_loss_orchestration/06.12_ema_prior_normalization_deep_dive.ipynb)) — tune the schedule against the normalized objective, not raw loss magnitudes.

## Section 6 — Further reading

- [`src/neural_data_decoding/training/schedules/factory.py`](../../src/neural_data_decoding/training/schedules/factory.py) — `make_weight_schedule`.
- [`src/neural_data_decoding/training/schedules/bundle.py`](../../src/neural_data_decoding/training/schedules/bundle.py) — `KLBaseAnneal` and the two-step KL pipeline.
- [06.4 EMA prior normalization](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb) — where the scheduled weights are consumed.

Next up: **[07.5 freeze / unfreeze curriculum](07.5_freeze_unfreeze_curriculum.ipynb)** — the third lever, and a surprise: freezing is *not* done with `requires_grad`.